<div align="center">

<a href="https://colab.research.google.com/github/utkukose/philosophy-of-ai-NB-lecture/blob/main/exams/final/FIN3_theory_of_mind_attribution.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Final Capstone: Minds, Morals and Machines

## Project 3: Theory of mind and misattribution: When is the intentional stance a real pattern?

**Philosophy of Artificial Intelligence (11118BLG003)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## The project

An observer infers goals by Bayesian inverse planning from the moves of four kinds of agents: goal-directed agents, random walkers, rule-following wall followers and chasers that pursue a moving target [1, 2]. The experiment measures how often goals are attributed and how many bits per move each stance needs to describe the behaviour, a compression reading of Dennett's real patterns [3].

**Required extensions.** Add a design stance that knows the rules of each agent, agents with changing goals, and observers with different priors. Identify the conditions under which attributing goals is a misattribution, and argue in the report how the results bear on attributing beliefs and desires to language model agents [4].

## What this example implements

The notebook simulates the four kinds of agents, computes the posterior probability of a goal against a no-goal hypothesis, and compares the description length per move under the intentional and the physical stance.

Run the cells in order. The example is a baseline: the capstone extends it, evaluates the extensions and reports the results.

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Four kinds of agents

In [ ]:
import pandas as pd
rng = np.random.default_rng(SEED)
N = 9
GOALS = {"A": (8, 8), "B": (8, 0), "C": (0, 8)}
ACTS = {"N": (0, 1), "E": (1, 0), "S": (0, -1), "W": (-1, 0)}

def legal(pos):
    return [(m, (pos[0] + d[0], pos[1] + d[1])) for m, d in ACTS.items() if 0 <= pos[0] + d[0] < N and 0 <= pos[1] + d[1] < N]

def goal_policy(pos, goal, beta=2.0):
    opts = legal(pos); u = np.array([-(abs(p[0] - goal[0]) + abs(p[1] - goal[1])) for _, p in opts]); w = np.exp(beta * (u - u.max()))
    return {m: wi for (m, _), wi in zip(opts, w / w.sum())}

def random_policy(pos):
    opts = legal(pos); return {m: 1 / len(opts) for m, _ in opts}

def move(pos, m):
    return (pos[0] + ACTS[m][0], pos[1] + ACTS[m][1])

def trajectory(kind, steps=12):
    pos, path, moves = (4, 4), [(4, 4)], []
    target, goal = (int(rng.integers(N)), int(rng.integers(N))), GOALS[str(rng.choice(list(GOALS)))]
    for t in range(steps):
        if kind == "goal-directed":
            pol = goal_policy(pos, goal)
        elif kind == "random walker":
            pol = random_policy(pos)
        elif kind == "wall follower":
            pol = {"E": 1.0} if pos[0] < N - 1 else ({"N": 1.0} if pos[1] < N - 1 else {"W": 1.0})
        else:
            opts = legal(target); target = move(target, opts[rng.integers(len(opts))][0]); pol = goal_policy(pos, target)
        m = str(rng.choice(list(pol), p=list(pol.values())))
        moves.append(m); pos = move(pos, m); path.append(pos)
    return path, moves

## 2. Attribution and description length

In [ ]:
def observer(path, moves, beta=2.0):
    logp = {g: np.log(0.25) for g in GOALS}; logp["no goal"] = np.log(0.25)
    ll = {"intentional": 0.0, "physical": 0.0}
    last = None
    for pos, m in zip(path[:-1], moves):
        v = np.array(list(logp.values())); post = np.exp(v - np.logaddexp.reduce(v)); post = dict(zip(logp, post))
        mix = {k: 0.0 for k, _ in legal(pos)}
        for g, pg in post.items():
            pol = random_policy(pos) if g == "no goal" else goal_policy(pos, GOALS[g], beta)
            for k, p in pol.items():
                mix[k] += pg * p
        ll["intentional"] += np.log2(mix[m])
        opts = [k for k, _ in legal(pos)]
        phys = {k: (0.7 if k == last else 0.3 / max(len(opts) - (last in opts), 1)) if last in opts else 1 / len(opts) for k in opts}
        ll["physical"] += np.log2(phys[m])
        for g in GOALS:
            logp[g] += np.log(goal_policy(pos, GOALS[g], beta)[m])
        logp["no goal"] += np.log(random_policy(pos)[m])
        last = m
    v = np.array(list(logp.values())); post = dict(zip(logp, np.exp(v - np.logaddexp.reduce(v))))
    return 1 - post["no goal"], {k: -x / len(moves) for k, x in ll.items()}

rows = []
for kind in ["goal-directed", "random walker", "wall follower", "chaser"]:
    for _ in range(150):
        pg, bits = observer(*trajectory(kind))
        rows.append(dict(agent=kind, attributed_goal=pg > 0.9, **{f"bits per move, {k} stance": v for k, v in bits.items()}))
pd.DataFrame(rows).groupby("agent").mean().round(3)

## Report and submission

Both tracks follow the course report template. A research report states its thesis in the introduction, reconstructs the relevant arguments as explicit premises and conclusions, and evaluates them. An application report explains what the simulation or experiment can and cannot show about the philosophical question, presents its results with figures and discusses its limitations. All sources are cited in square brackets.

This capstone also supports self-learning and can be completed at any pace. When the course is taught actively in a semester, the final capstone is sent by e-mail to utkukose@sdu.edu.tr or utkukose@gmail.com no later than 23:53 (Türkiye time) on the last Sunday of Week 14, with the report and all code files attached or linked.

## References

[1] Baker, C. L., Saxe, R., & Tenenbaum, J. B. (2009). Action understanding as inverse planning. *Cognition*, 113(3), 329-349. <https://doi.org/10.1016/j.cognition.2009.07.005>

[2] Heider, F., & Simmel, M. (1944). An experimental study of apparent behavior. *The American Journal of Psychology*, 57(2), 243-259. <https://doi.org/10.2307/1416950>

[3] Dennett, D. C. (1991). Real patterns. *The Journal of Philosophy*, 88(1), 27-51. <https://doi.org/10.2307/2027085>

[4] Shanahan, M. (2024). Talking about large language models. *Communications of the ACM*, 67(2), 68-79. <https://doi.org/10.1145/3624724>